# Sales Visualization Lab

This notebook is a guided introduction to Matplotlib, Seaborn, and Plotly using the fictional `data/sales.csv` dataset. Run cells in order. It is designed for learning and exploration; the complete script in `src/visualize_sales.py` creates 32 chart files.

## 1. Load the data and inspect data quality

Check row count, column types, date range, missing values, and duplicate order IDs before plotting.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd()
# If the notebook was launched from another folder, find the project root.
if not (ROOT / "data" / "sales.csv").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "sales.csv", parse_dates=["order_date"])
df["month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
print("Shape:", df.shape)
display(df.head())
df.info()

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Basic data-quality checks
print("Missing values per column:\n", df.isna().sum())
print("Duplicate order IDs:", df["order_id"].duplicated().sum())
print("Date range:", df["order_date"].min().date(), "to", df["order_date"].max().date())
display(df[["quantity", "unit_price", "discount", "sales", "profit"]].describe().round(2))

## 2. Matplotlib: line, bar, scatter and histogram

Use line charts for time trends, bars for comparisons, scatter plots for relationships, and histograms for distributions.

In [ ]:
monthly = df.groupby("month", as_index=False).agg(sales=("sales", "sum"), profit=("profit", "sum"))
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(monthly["month"], monthly["sales"], marker="o")
axes[0, 0].set(title="Monthly Sales", xlabel="Month", ylabel="Sales (₹)")
cat_sales = df.groupby("category")["sales"].sum().sort_values(ascending=False)
axes[0, 1].bar(cat_sales.index, cat_sales.values)
axes[0, 1].set(title="Sales by Category", xlabel="Category", ylabel="Sales (₹)")
axes[0, 1].tick_params(axis="x", rotation=25)
axes[1, 0].scatter(df["sales"], df["profit"], alpha=0.4)
axes[1, 0].axhline(0, linestyle="--")
axes[1, 0].set(title="Sales vs Profit", xlabel="Sales (₹)", ylabel="Profit (₹)")
axes[1, 1].hist(df["sales"], bins=30, edgecolor="white")
axes[1, 1].set(title="Sales Distribution", xlabel="Sales per Order (₹)", ylabel="Order Count")
plt.tight_layout()
plt.show()

## 3. Matplotlib: pie chart and subplots

Pie charts work best with a small number of groups. Use subplots to compare related measures while giving each measure a clear label.

In [ ]:
segment_sales = df.groupby("customer_segment")["sales"].sum()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(segment_sales.values, labels=segment_sales.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Sales Share by Customer Segment")
axes[1].plot(monthly["month"], monthly["profit"], marker="s", label="Profit")
axes[1].set(title="Monthly Profit", xlabel="Month", ylabel="Profit (₹)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Seaborn: distributions, categories and correlations

Seaborn adds statistical plotting shortcuts and attractive defaults. A correlation heatmap describes association, not causation.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="sales", bins=30, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Sales Histogram with KDE")
sns.boxplot(data=df, x="region", y="profit", showfliers=False, color="steelblue", ax=axes[1])
axes[1].axhline(0, linestyle="--", linewidth=1)
axes[1].set_title("Profit Spread by Region")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[["quantity", "unit_price", "discount", "sales", "profit"]].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# A pair plot compares numeric variables two at a time; sample rows to keep it readable.
sample = df[["quantity", "discount", "sales", "profit", "category"]].sample(min(300, len(df)), random_state=42)
sns.pairplot(sample, vars=["quantity", "discount", "sales", "profit"], hue="category", corner=True, plot_kws={"alpha": 0.5, "s": 18})

## 5. Seaborn: regression and violin plot

A regression plot can reveal an approximate linear relationship. A violin plot shows the shape of each group's distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=df.sample(min(700, len(df)), random_state=3), x="sales", y="profit", color="steelblue", scatter_kws={"alpha": 0.3, "s": 18}, ax=axes[0])
axes[0].axhline(0, linestyle="--", linewidth=1)
axes[0].set_title("Sales vs Profit with Regression")
sns.violinplot(data=df, x="category", y="profit", inner="quartile", cut=0, color="steelblue", ax=axes[1])
axes[1].set_title("Profit Distribution by Category")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 6. Plotly: interactive line and scatter plots

Hover over points for detail, zoom, pan, or click legend items to show/hide categories.

In [ ]:
fig = px.line(monthly, x="month", y="sales", markers=True, title="Interactive Monthly Sales", labels={"month": "Month", "sales": "Sales (₹)"}, template="plotly_white")
fig.show()

In [ ]:
fig = px.scatter(df.sample(min(800, len(df)), random_state=5), x="sales", y="profit", color="category", size="quantity", hover_data=["order_id", "region", "discount", "channel"], title="Interactive Sales vs Profit", labels={"sales": "Sales (₹)", "profit": "Profit (₹)"}, template="plotly_white")
fig.add_hline(y=0, line_dash="dash")
fig.show()

## 7. Plotly: heatmap and 3D scatter

3D charts are useful for exploration but can be harder to read accurately than 2D charts. Use them only when the third dimension adds real value.

In [ ]:
pivot = df.pivot_table(index="category", columns="channel", values="sales", aggfunc="sum", fill_value=0)
px.imshow(pivot, text_auto=".2s", aspect="auto", title="Sales by Category and Channel", labels={"x": "Channel", "y": "Category", "color": "Sales (₹)"}).show()

three_d = df.sample(min(400, len(df)), random_state=6)
px.scatter_3d(three_d, x="sales", y="profit", z="discount", color="category", hover_data=["order_id", "region"], title="3D Sales / Profit / Discount").show()

## 8. Export an interactive chart to HTML

HTML exports are useful when you want to share Plotly interactivity without running a dashboard server.

In [ ]:
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
fig = px.bar(df.groupby("region", as_index=False)["sales"].sum(), x="region", y="sales", title="Sales by Region", labels={"sales": "Sales (₹)"}, template="plotly_white")
html_path = out_dir / "notebook_sales_by_region.html"
fig.write_html(html_path, include_plotlyjs="directory")
print("Saved interactive chart to:", html_path)

## 9. Dashboard and best-practice checklist

Run the Streamlit dashboard from the project root in PowerShell with `streamlit run src/dashboard.py`. Before publishing a chart, check:

- Does the chart answer a specific question?
- Are title, axes, units and filters clear?
- Is the chart type suitable for comparison, time trends, distributions or relationships?
- Are colors consistent and readable?
- Could a truncated axis, overloaded legend, or 3D perspective mislead?
- Are statistical intervals and limitations described accurately?

### Mini-exercise
Find the category with the highest total sales, the region with the highest total profit, and whether larger discounts appear to be associated with lower profit. Write a short evidence-based conclusion and include the timeframe and units.